In [1]:
print(123)

123


In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if the Zoom link isn’t shared with students?',
 'document': '489dd1c9d9'}

In [6]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [9]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join now. It’s not too late.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [10]:
assistant.total_cost()

0.00056475

In [11]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [12]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course — can I still join it now, or is it too late?',
 'answer_llm': 'Yes — you can still join now. It’s not too late.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [13]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [14]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course — can I still join it now, or is it too late?',
 'answer_llm': 'Yes, you can still join it now. It’s not too late.\n\nIf you want to receive a certificate, make sure you submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [15]:
assistant.reset_usage()

In [16]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [17]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/425 [00:00<?, ?it/s]

In [18]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [19]:
assistant.total_cost()

0.3809332500000002

In [20]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [22]:
df_answers.head()

,question,answer_llm,answer_orig,document
0,I just found this course — can I still join it...,Yes — you can still join it now. \nIf you wan...,"Yes, but if you want to receive a certificate,...",74eb249bbf
1,"If I enroll late, will I still be able to get ...","Yes, but only if you submit your capstone proj...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,Do I have to submit the project before submiss...,"Yes. To qualify for the certificate, you need ...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Is it okay to start the course after it has al...,Yes — you can start the course whenever you wa...,"Yes, but if you want to receive a certificate,...",74eb249bbf
4,What’s the deadline for project submission if ...,You need to submit your project while the cour...,"Yes, but if you want to receive a certificate,...",74eb249bbf
